MAGALY BENAVIDES SANTIAGO

In [70]:
#Importamos bibliotecas
import pandas as pd
import numpy as np
from  pathlib import Path

In [71]:
#Rutas que usaremos
ruta_raw= Path("data/raw")
ruta_processed = Path("data/processed")
archivo_raw = ruta_raw / "SPY_2015_2025_raw.csv"
archivo_clean = ruta_processed / "SPY_2015_2025_clean.csv"
archivo_bitacora = ruta_processed / "bitacora_limpieza_SPY.csv"

In [72]:
#Carpeta de salida
ruta_processed.mkdir(parents=True, exist_ok=True)

In [73]:
#Comprobamos que existe el archivo creado en semana 3 
if not archivo_raw.exists(): raise FileNotFoundError(f"No se encontro el archivo: {archivo_raw}")

In [87]:
#Recuperamos archivo
datos_raw = pd.read_csv(archivo_raw, index_col=0, parse_dates=True)
if datos_raw.empty: raise ValueError("La base original esta vacia.")

In [76]:
#Analizamos su estructura
print(datos_raw.shape)
print(datos_raw.columns)
print(type(datos_raw.index))
datos_raw.head()

(2766, 6)
Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2015-01-02,169.687836,205.429993,206.880005,204.179993,206.380005,121465900
2015-01-05,166.623337,201.720001,204.369995,201.350006,204.169998,169632600
2015-01-06,165.053909,199.820007,202.720001,198.860001,202.089996,209151400
2015-01-07,167.110703,202.309998,202.720001,200.880005,201.419998,125346700
2015-01-08,170.076080,205.899994,206.160004,203.990005,204.009995,147217800


CREACION DE UNA COPIA DE TRABAJO

In [7]:
datos = datos_raw.copy(deep=True)

In [8]:
print("Mismo objeto:", datos is datos_raw)
print("Dimensiones raw:", datos_raw.shape)
print("Dimensiones copia:", datos.shape)

Mismo objeto: False
Dimensiones raw: (2766, 6)
Dimensiones copia: (2766, 6)


In [9]:
#Se crea bitacora 
bitacora = []
def registrar_cambio(paso, problema, accion, filas_afectadas):bitacora.append({
"Paso": paso,
"Problema detectado": problema,
"Accion aplicada": accion,
"Filas afectadas": int(filas_afectadas)})

In [10]:
#Indica si valido la bitacora
registrar_cambio(paso="Lectura", problema="Ninguno", accion="Se creo una copia de trabajo", filas_afectadas=0)

In [11]:
bitacora_df = pd.DataFrame(bitacora)
bitacora_df

,Paso,Problema detectado,Accion aplicada,Filas afectadas
0,Lectura,Ninguno,Se creo una copia de trabajo,0


NORMALIZACION DEL INDICE TEMPORAL

In [12]:
datos.index = pd.to_datetime(datos.index, errors="coerce")

In [13]:
fechas_invalidas = datos.index.isna().sum()
print("Fechas invalidas:", fechas_invalidas)

Fechas invalidas: 0


In [14]:
#Elimina fechas invalidas en caso de que haya
if fechas_invalidas > 0:
    print(datos.loc[datos.index.isna()])

In [15]:
if datos.index.tz is not None: datos.index = datos.index.tz_localize(None)

In [16]:
#Se le pone nombre al indice
datos.index.name = "Date"

ORDENAMIENTO CRONOLOGICO

In [17]:
estaba_ordenada = datos.index.is_monotonic_increasing
print("La base estaba ordenada:", estaba_ordenada)

La base estaba ordenada: True


In [18]:
datos = datos.sort_index()

In [19]:
registrar_cambio(paso="Orden temporal", problema=("Ninguno" if estaba_ordenada
else "Las fechas no estaban ordenadas"), accion="Se ordeno la base por fecha", filas_afectadas=(0 if estaba_ordenada else len(datos)))

In [20]:
#Verificacion
assert datos.index.is_monotonic_increasing

FECHAS DUPLICADAS

In [21]:
filas_duplicadas_exactas = datos.reset_index().duplicated().sum()
print( "Filas completamente duplicadas:", filas_duplicadas_exactas)

Filas completamente duplicadas: 0


In [22]:
mascara_fechas_duplicadas = datos.index.duplicated(keep=False)
fechas_duplicadas = datos.loc[mascara_fechas_duplicadas]
print("Filas con fecha repetida:", len(fechas_duplicadas))

Filas con fecha repetida: 0


REVISION DE COLUMNAS

In [23]:
columnas_esperadas = ["Open","High", "Low","Close", "Adj Close","Volume"]

In [24]:
columnas_faltantes = [columna
for columna in columnas_esperadas
if columna not in datos.columns]
columnas_extra = [columna
for columna in datos.columns
if columna not in columnas_esperadas]
print("Columnas faltantes:", columnas_faltantes)
print("Columnas adicionales:", columnas_extra)

Columnas faltantes: []
Columnas adicionales: []


In [25]:
datos = datos[columnas_esperadas].copy()

CONVERSION DE COLUMNAS NUMERICAS

In [26]:
for columna in columnas_esperadas:datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

In [27]:
print(datos.dtypes)

Open         float64
High         float64
Low          float64
Close        float64
Adj Close    float64
Volume         int64
dtype: object


VALORES FALTANTES

In [28]:
#creamos un reporte
reporte_faltantes = pd.DataFrame({"Valores faltantes": datos.isna().sum(),
"Porcentaje": 100 * datos.isna().mean()})
reporte_faltantes

,Valores faltantes,Porcentaje
Open,0,0.0
High,0,0.0
Low,0,0.0
Close,0,0.0
Adj Close,0,0.0
Volume,0,0.0


In [29]:
filas_con_faltantes = datos[datos.isna().any(axis=1)]
print("Filas con al menos un faltante:", len(filas_con_faltantes))

Filas con al menos un faltante: 0


DIAS NO BURSATILES Y FECHAS AUSENTES

In [30]:
mascara_fin_semana = datos.index.dayofweek >= 5 
numero_fin_semana = mascara_fin_semana.sum()
print("Observaciones en fin de semana:", numero_fin_semana)

Observaciones en fin de semana: 0


In [31]:
diferencias = datos.index.to_series().diff().dt.days
print(diferencias.value_counts().sort_index())

1.0    2164
2.0      27
3.0     498
4.0      76
Name: Date, dtype: int64


VALIDACION DE PRECIOS Y VOLUMEN

In [32]:
columnas_precios = [
 "Open",
 "High",
 "Low",
 "Close",
 "Adj Close"]
mascara_precios_no_positivos = (datos[columnas_precios] <= 0).any(axis=1)
numero_precios_no_positivos = (mascara_precios_no_positivos.sum())

In [33]:
if numero_precios_no_positivos > 0: 
    display(datos.loc[mascara_precios_no_positivos])
    raise ValueError(
    "Existen precios no positivos. "
    "No se pueden calcular logaritmos.")

RELACIONES OHLC

In [34]:
mascara_ohlc = (
(datos["High"] < datos["Low"])
 | (datos["Open"] < datos["Low"])
 | (datos["Open"] > datos["High"])
 | (datos["Close"] < datos["Low"])
 | (datos["Close"] > datos["High"]))
numero_inconsistencias_ohlc = mascara_ohlc.sum()

In [35]:
if numero_inconsistencias_ohlc > 0: 
    display(datos.loc[mascara_ohlc])
    raise ValueError(
      "Existen inconsistencias OHLC. "
       "Deben verificarse contra la fuente."
    )

In [36]:
mascara_volumen_negativo = datos["Volume"] < 0
numero_volumen_negativo = mascara_volumen_negativo.sum()
if numero_volumen_negativo > 0:
    display(datos.loc[mascara_volumen_negativo])
    raise ValueError(
     "Existen observaciones con volumen negativo."
 )

REGISTRO DE VALIDACION

In [37]:
registrar_cambio(paso="Validacion financiera", problema="Precios, relaciones OHLC y volumen", accion="Se verificaron las condiciones basicas",
filas_afectadas=(numero_precios_no_positivos + numero_inconsistencias_ohlc + numero_volumen_negativo))

OBSERVACIONES EXTREMAS

In [38]:
datos.describe().T

,count,mean,std,min,25%,50%,75%,max
Open,2766.0,3.607312e+02,1.321377e+02,1.823400e+02,2.479650e+02,3.264600e+02,4.437025e+02,6.906400e+02
High,2766.0,3.626776e+02,1.327516e+02,1.841000e+02,2.497775e+02,3.279350e+02,4.455050e+02,6.916600e+02
Low,2766.0,3.586015e+02,1.314180e+02,1.810200e+02,2.470100e+02,3.237700e+02,4.416775e+02,6.892700e+02
Close,2766.0,3.607912e+02,1.321496e+02,1.828600e+02,2.480800e+02,3.266550e+02,4.435650e+02,6.903800e+02
Adj Close,2766.0,3.361447e+02,1.399862e+02,1.541616e+02,2.183512e+02,2.991299e+02,4.208833e+02,6.867305e+02
Volume,2766.0,8.608106e+07,4.392519e+07,2.027000e+07,5.874842e+07,7.570780e+07,9.941975e+07,5.072443e+08


In [39]:
datos["Volume"].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])

0.50     75707800.0
0.75     99419750.0
0.90    135530600.0
0.95    165855075.0
0.99    257911470.0
1.00    507244300.0
Name: Volume, dtype: float64

In [55]:
def auditar_base_financiera(df):
    columnas = [
         "Open",
         "High",
         "Low",
         "Close",
         "Adj Close",
         "Volume"
]
    reporte = {
     "Numero de filas": len(df),
     "Numero de columnas": df.shape[1],
     "Indice temporal": isinstance(
         df.index,
         pd.DatetimeIndex
    ),
    "Fechas ordenadas": (
         df.index.is_monotonic_increasing
    ),
    "Fechas unicas": df.index.is_unique,
    "Total de faltantes": int(
       df[columnas].isna().sum().sum()
    ),
    "Precios no positivos": int(
       (df[[
      "Open", "High", "Low",
      "Close", "Adj Close"
    ]] <= 0).sum().sum()
   ),
  "Volumen negativo": int(
    (df["Volume"] < 0).sum()
   ),
    "Inconsistencias OHLC": int((
     (df["High"] < df["Low"])
     | (df["Open"] < df["Low"])
     | (df["Open"] > df["High"])
     | (df["Close"] < df["Low"])
     | (df["Close"] > df["High"])
    ).sum()),
      "Observaciones en fin de semana": int(
        (df.index.dayofweek >= 5).sum()
     )
    }

    return pd.Series(
         reporte,
         name="Resultado"
       ).to_frame()

In [57]:
auditoria_raw = auditar_base_financiera(datos_raw)
auditoria_clean = auditar_base_financiera(datos)

In [58]:
comparacion_auditoria = pd.concat(
 [auditoria_raw, auditoria_clean],
 axis=1
)
comparacion_auditoria.columns = [
     "Base original",
      "Base limpia"
]
comparacion_auditoria

,Base original,Base limpia
Numero de filas,2766,2766
Numero de columnas,6,6
Indice temporal,True,True
Fechas ordenadas,True,True
Fechas unicas,True,True
Total de faltantes,0,0
Precios no positivos,0,0
Volumen negativo,0,0
Inconsistencias OHLC,0,0
Observaciones en fin de semana,0,0


RESUMEN DE CAMBIOS

In [60]:
resumen_limpieza = pd.DataFrame({
 "Elemento": [
     "Filas originales",
     "Filas procesadas",
     "Filas eliminadas",
     "Columnas originales",
     "Columnas procesadas",
     "Primera fecha original",
     "Ultima fecha original",
     "Primera fecha procesada",
     "Ultima fecha procesada"
 ],
 "Resultado": [
     len(datos_raw),
     len(datos),
     len(datos_raw) - len(datos),
     datos_raw.shape[1],
     datos.shape[1],
     datos_raw.index.min(),
     datos_raw.index.max(),
     datos.index.min(),
     datos.index.max()
 ]
})
resumen_limpieza

,Elemento,Resultado
0,Filas originales,2766
1,Filas procesadas,2766
2,Filas eliminadas,0
3,Columnas originales,6
4,Columnas procesadas,6
5,Primera fecha original,2015-01-02 00:00:00
6,Ultima fecha original,2025-12-31 00:00:00
7,Primera fecha procesada,2015-01-02 00:00:00
8,Ultima fecha procesada,2025-12-31 00:00:00


CONDICIONES MINIMAS DE ACEPTACION

In [62]:
assert isinstance(datos.index, pd.DatetimeIndex)
assert datos.index.is_monotonic_increasing
assert datos.index.is_unique
assert (datos[columnas_precios] > 0).all().all()
assert (datos["Volume"] >= 0).all()
assert not mascara_ohlc.any()
assert (datos.index.dayofweek < 5).all()

EXPORTACION DE LA BASE PROCESADA

In [63]:
datos.to_csv(
 archivo_clean,
 index=True,
 date_format=" %Y- %m- %d"
 )

print("Base limpia guardada en:", archivo_clean)

Base limpia guardada en: data\processed\SPY_2015_2025_clean.csv


In [88]:
#Bitacora guardad de manera separada
bitacora_df = pd.DataFrame(bitacora)
bitacora_df.to_csv(
archivo_bitacora,
index=False)
print('Base limpia:', archivo_bitacora)

Base limpia: data\processed\bitacora_limpieza_SPY.csv


In [89]:
#Se guarda unn resumen
archivo_resumen = (ruta_processed / "resumen_limpieza_SPY.csv" )
resumen_limpieza.to_csv(archivo_resumen, index=False )
print('Resumen:', archivo_resumen)

Resumen: data\processed\resumen_limpieza_SPY.csv


RECUPERACION Y VALIDACION DEL ARCHIVO LIMPIO

In [82]:
#Guardamos y leemos el archivo nuevamente
datos_verificados = pd.read_csv(archivo_clean, index_col="Date", parse_dates=True)

In [83]:
#Comprobacion de que se guardo correctamente
assert datos_verificados.shape == datos.shape
assert datos_verificados.columns.tolist() == datos.columns.tolist()
assert datos_verificados.index.equals(datos.index)

In [85]:
pd.testing.assert_frame_equal( datos_verificados, datos, check_freq=False,
check_dtype=False, rtol=1e-10, atol=1e-12)

In [86]:
print("El archivo limpio se guardo y recupero " "correctamente.")

El archivo limpio se guardo y recupero correctamente.


# CONCLUSION

Durante esta semana se realizó la limpieza, validación y preparación de la base histórica de SPY correspondiente al periodo 2015–2025. La base original contenía 2,766 filas y 6 columnas, y después del proceso de limpieza se conservaron 2,766 filas y 6 columnas, por lo que no fue necesario eliminar ninguna fila.
La revisión de duplicados mostró 0 filas completamente duplicadas y 0 filas con fechas repetidas. No se encontraron columnas faltantes ni adicionales. La revisión de valores faltantes mostró 0 valores faltantes en las variables analizadas y 0 filas con al menos un faltante.
La cobertura temporal se mantuvo sin cambios, desde el 2 de enero de 2015 hasta el 31 de diciembre de 2025. También se verificó que no existieran observaciones correspondientes a fines de semana.
Se validaron las relaciones entre los precios open, high, low y close, así como el volumen, mediante las condiciones establecidas en el proceso de validación. No se generaron errores durante estas comprobaciones. También se revisaron las observaciones extremas sin eliminarlas automáticamente, ya que un valor extremo puede corresponder a un movimiento real del mercado.
Finalmente, la base limpia fue exportada junto con su bitácora de limpieza y resumen de cambios. El archivo limpio fue leído nuevamente y comparado con la base procesada, comprobándose que se guardó y recuperó correctamente.